In [1]:
import cv2
import torch
from facenet_pytorch import MTCNN,InceptionResnetV1
from torchvision import transforms
from PIL import Image
from arcface_loss import ArcFaceLoss
import pickle

/home/shk/Documents/projects/python/shkreco/face_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
with open('worker_db.pkl', 'rb') as f:
    worker_db = pickle.load(f)
list(keys) ,embeddings = worker_db.keys(), worker_db.values()

In [4]:
checkpoint = torch.load('checkpoints/arcface_best.pth', 
                       map_location=torch.device('cpu'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recreate backbone and loss modules
backbone = InceptionResnetV1(pretrained='vggface2').to(device)
arc_loss = ArcFaceLoss(
    embedding_dim=512,
    num_classes=len(checkpoint['classes']),
    s=64.0,
    m=0.5,
).to(device)

# Load weights
backbone.load_state_dict(checkpoint['backbone'])
arc_loss.load_state_dict(checkpoint['arc_loss'])

# Switch to eval mode
backbone.eval()
arc_loss.eval()

# Prepare image
transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

In [ ]:
# Get embedding for an image
with torch.no_grad():
    image = Image.open('data/train/Alejandro_Toledo/Alejandro_Toledo_0004.jpg').convert('RGB')
    image = transform(image).unsqueeze(0).to(device)
    embedding = backbone(image)  # Shape: (1, 512)

print(f"Embedding shape: {embedding.shape}")
print(f"Classes: {checkpoint['classes']}")
print(f"Trained on epoch: {checkpoint['epoch']}")
# %%


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


Embedding shape: torch.Size([1, 512])
Classes: ['Alejandro_Toledo', 'Alvaro_Uribe', 'Andre_Agassi', 'Ariel_Sharon', 'Arnold_Schwarzenegger', 'Colin_Powell', 'David_Beckham', 'Donald_Rumsfeld', 'George_W_Bush', 'Gerhard_Schroeder']
Trained on epoch: 55


In [ ]:
img = Image.open('data/train/George_W_Bush/George_W_Bush_0006.jpg').convert('RGB')
mtcnn   = MTCNN(image_size=160, margin=20, keep_all=False, device=device)
face_tensor = mtcnn(img)
if face_tensor is not None:
    face_tensor = face_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        emb = backbone(face_tensor)
    for i, embedding in enumerate(embeddings):
        embedding = torch.tensor(embedding).unsqueeze(0).to(device)
        similarity = torch.nn.functional.cosine_similarity(embedding, emb)
        print(f"{keys[i]} Similarity: {similarity.item()}")
        if similarity.item() > 0.6:  # Adjust threshold as needed
            print("Same person")
        else:
            print("Different person")

TypeError: 'dict_keys' object is not subscriptable

In [ ]:
image1 = Image.open("images/ism/01.png").convert('RGB')
image2 = Image.open("images/shms/02.png").convert('RGB')

image1 = transform(image1).unsqueeze(0).to(device)
image2 = transform(image2).unsqueeze(0).to(device)
# Get embeddings for two faces
emb1 = backbone(image1)  # (1, 512)
emb2 = backbone(image2)  # (1, 512)

# Compute cosine similarity
similarity = torch.nn.functional.cosine_similarity(emb1, emb2)
print(f"Similarity score: {similarity.item():.4f}")  # Higher = more similar

# Threshold-based matching
if similarity.item() > 0.6:  # Adjust threshold as needed
    print("Same person")
else:
    print("Different person")

In [ ]:

# %%

    
# %%

# %%
import os
import matplotlib.pyplot as plt
base = Image.open("data/train/George_W_Bush/George_W_Bush_0006.jpg").convert('RGB')
bases = os.listdir('data/train/George_W_Bush/')

tests = os.listdir('data/test/George_W_Bush/')
for base in bases:
    print(f"Base: {base}")
    print("-" * 30)
    base = Image.open(f"data/train/George_W_Bush/{base}").convert('RGB')
    boxes, _ = MTCNN().detect(base)
    if boxes is not None:
        x1, y1, x2, y2 = boxes[0]
        base = base.crop((x1, y1, x2, y2))
    base = transform(base).unsqueeze(0).to(device)
    for test in tests:
        img = Image.open(f"data/test/George_W_Bush/{test}").convert('RGB')
        boxes, _ = MTCNN().detect(img)
        if boxes is not None:
            x1, y1, x2, y2 = boxes[0]
            img = img.crop((x1, y1, x2, y2))
        img = transform(img).unsqueeze(0).to(device)
        emb1 = backbone(base)
        emb2 = backbone(img)
        similarity = torch.nn.functional.cosine_similarity(emb1, emb2)
        print(f"{test}: Similarity score: {similarity.item():.4f}")
# %%
